<a href="https://colab.research.google.com/github/peedrocs/estudando-RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breve estudo para compreensão da implementação e do funcionamento de um RAG (Retrival Augmented Generation)

## Célula 1 - Bibliotecas

- sentence-transformers -> transforma os textos em vetores numéricos
- transformers -> biblioteca capaz de rodar LLMs
- accelerate -> auxilia no funcionamento de LLMs grandes
- bitsandbytes -> permite carregar modelos de LLMs comprimidos utilizando menos memória
- datasets -> biblioteca para baixar e manipular dados do Hugging Face

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes
!pip install -q datasets


## Célula 2 - Baixando base de dados

Conjunto de dados que o RAG vai buscar informação para responder as perguntas.

- load_dataset() -> baixa o dataset
- split=train[:200] pega apenas as 200 primeiras linhas para não sobrecarregar o colab
Nesse data set, cada linha tem um paragrafo de texto, uma pergunta e a resposta esperada
- dataset['context'] pega apenas o paragrafo de texto

In [ ]:
from datasets import load_dataset

dataset = load_dataset("nunorc/squad_v1_pt", split="train[:200]")

documentos = list(set(dataset['context']))
print(f"Total de documentos: {len(documentos)}")

Total de documentos: 41


## Célula 3 - Geração dos embeddings
Transforma os textos em um vetor numérico que representa a significância daquele texto

- torch.cuda.is_available(): verifica se tem GPU disponível, se tiver ela é usada, caso contrário utiliza-se a CPU
- SentenceTransformer: utiliza um LLM menor para transformar as frases em embeddings
modelo_embeddings.encode(): transforma todos os documentos em embeddings
- embeddings_docs.shape: mostra as dimensões dos vetores gerados (qtd_docs, tam_vet)



In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando {device}")

modelo_embeddings = SentenceTransformer('all-MiniLM-L6-v2', device=device)
embeddings_docs = modelo_embeddings.encode(documentos, convert_to_numpy=True)
print(embeddings_docs.shape)

Usando cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(41, 384)


## Célula 4 - Buscando semelhança
### Método utilizado: Similaridade de cosseno

Foi utilizado esse método por ser mais adequado para mensurar a semelhança semântica entre vetores de gerados de textos

- np.dot(): gera o produto escalar entre dois vetores
- np.linalg.norm(): calcula a magnitude do vetor

O resultado da divisão representa a similaridade entre eles quanto mais próximo a 1 mais parecidos eles são e quanto mais próximo de -1 mais diferentes eles são.

In [ ]:
def similaridade_cosseno(a,b):
  return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

## Célula 5 - Função de busca/recuperação

Com base em uma pergunta ele procura os dados mais relevantes na base disponível

- modelo_embeddings.encode(): Transforma a pergunta feita no vetor numérico
- Em seguida é calculada a similaridade da pergunta com cada um dos elementos na lista de documentos
- np.argsort(): ordena os valores do menos parecido(<0) para o mais parecido(>0).
- Pega os 2 primeiros resultados por padrão e devolve uma lista de tuplas com o texto do documento e seu nível de proximidade.

In [ ]:
def buscar(pergunta, top_k=2):
  emb_pergunta = modelo_embeddings.encode(pergunta, convert_to_numpy=True)
  scores = [similaridade_cosseno(emb_pergunta, emb_doc) for emb_doc in embeddings_docs]
  indices_ordenados = np.argsort(scores)[::-1][:top_k]
  return [(documentos[i], scores[i]) for i in indices_ordenados]

### Célula 6 - Apenas um teste para confirmar o funcionamento da função de busca

In [ ]:
for doc, score in buscar("Em qual país e cidade fica a Catedral de Notre Dame"):
  print(f"{score:.3f} - {doc}")

0.620 - Santa Cruz O padre John Francis O&#39;Hara foi eleito vice-presidente em 1933 e presidente da Notre Dame em 1934. Durante seu mandato em Notre Dame, ele trouxe numerosos intelectuais de refugiados para o campus; ele selecionou Frank H. Spearman, Jeremiah DM Ford, Irvin Abell e Josephine Brownson para a Medalha Laetare, instituída em 1883. O&#39;Hara acreditava fortemente que o time de futebol Fighting Irish poderia ser um meio eficaz de &quot;familiarizar o público com os ideais&quot;. que dominam &quot;Notre Dame. Ele escreveu: &quot;O futebol de Notre Dame é um serviço espiritual porque é jogado pela honra e glória de Deus e de sua Mãe Santíssima. Quando São Paulo disse: &#39;Se você come ou bebe, ou qualquer outra coisa que você faz, faça tudo por a glória de Deus &quot;, ele incluiu futebol.&quot;
0.602 - Em 1919, o padre James Burns tornou-se presidente da Notre Dame e, em três anos, produziu uma revolução acadêmica que elevou a escola aos padrões nacionais, adotando o sis

## Célula 7 - Importando o modelo de geração

Ele carrega o modelo de linguagem que será utilizado para gerar as respostas

- AutoTokenizer: É o tradutor que converte o texto em tokens para o modelo poder consumir
- AutoModelForCausalLM: É o modelo de linguagem em si, nesse estudo foi utilizado o Qwen2.5 que possui 1.5 bilhão de paramêtros.
- torch_dtype=torch.float16: reduz o tamanho dos números utilizado pelo modelo para 16bits, visando economizar memória.
- device_map="auto": Serve para decidir se irá utilizar a GPU ou CPU



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

nome_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
modelo = AutoModelForCausalLM.from_pretrained(
    nome_modelo,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Célula 8 - Geração da Resposta

A função recebe um prompt e retorna a resposta gerada pela LLM

- mensagens: Padroniza o input do prompt como se fosse um usuário digitando no chat.
- tokenizer.apply_chat_template(): Formata a mensagem seguindo o modelo específico que o Qwen espera receber.
- tokenizer(): Transforma o texto em tokens para o modelo consumir, no formato de tensor do Pytorch (framework utilizado pelo Qwen)
- .to(): Assegura que os dados estão no mesmo local que a LLM (CPU ou GPU)
- wit torch.no_grad(): Serve para acelerar o processo de geração de texto, pois desativa o calculo dos gradientes que é necessário para o treinamento.
- modelo.generate(): É quem de fato gera a resposta
  -  max_new_tokens=300: Limita o tamanho da resposta gerada
  - do_sample=False: Torna o modelo mais determinístico, evitando a "critividade", consequentemente as alucinações
  - pad_token_id=tokenizer.eos_token_id: Atribui quais tokens vão preencher os espaços em branco.
- tokenizer.decode(): Converte os números gerados para texto novamente

In [ ]:
def gerar_resposta(prompt, max_tokens=300):
  mensagens = [{"role": "user", "content": prompt}]
  texto_formatado = tokenizer.apply_chat_template(
      mensagens, tokenize=False, add_generation_prompt=True
  )
  inputs = tokenizer(texto_formatado, return_tensors="pt").to(modelo.device)

  with torch.no_grad():
    saida = modelo.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
  resposta = tokenizer.decode(saida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
  return resposta

## Célula 9 - RAG em ação
- buscar(): Chamada da função para encontrar os documentos relevantes para a pergunta feita
- contexto_texto: transforma a lista dos documentos em um único bloco de texto
- O prompt dá a instrução do que o modelo deve fazer caso tenha material suficiente para responder e caso não possua o que ele deve fazer
- gerar_resposta(): Manda o prompt completo para a LLM e gera a resposta final

In [ ]:
def resposta_com_RAG(pergunta, top_k=2):
  contexto_recuperado = buscar(pergunta,top_k=top_k)
  contexto_texto = "\n".join([f"- {doc}" for doc, _ in contexto_recuperado])

  prompt = f"""Use o contexto abaixo para responder a pergunta de forma clara e direta. Se a resposta não estiver no contexto, diga que não sabe.

  Contexto:
  {contexto_texto}

  Pergunta: {pergunta}"""

  return gerar_resposta(prompt)

resposta = resposta_com_RAG("A universidade de Notre Dame é conhecida por quais cursos?")
print(resposta)

A universidade de Notre Dame é conhecida principalmente por seus programas de graduação e pós-graduação em áreas como:

1. Arquitetura: É reconhecida internacionalmente pelo seu programa de arquitetura, especialmente pela sua abordagem à "Nova Arquitetura Clássica".

2. Direito: Possui uma Faculdade de Direito notável, com programas de mestrado e doutorado bem avaliados.

3. Economia e Negócios: Oferece programas de graduação e pós-graduação em economia e negócios.

4. Engenharia: Tem programas de engenharia reconhecidos, incluindo engenharia civil, eletrônica e computação.

5. Ciências: Oferece programas de graduação e pós-graduação em diversas áreas de ciências, como biologia, física, matemática e química.

6. Artes e Letras: Possui programas de graduação e pós-graduação em áreas como literatura, história, filosofia e música.

É importante notar que além desses cursos principais, a universidade oferece uma ampla gama de opções de graduação e pós-graduação em várias outras áreas, mas 

In [ ]:
for i in range(5):
  print(f"{i}: {dataset[i]['question']}")

0: A quem a Virgem Maria supostamente apareceu em 1858 em Lourdes, na França?
1: O que fica em frente ao edifício principal de Notre Dame?
2: A Basílica do Sagrado Coração em Notre Dame fica ao lado de qual estrutura?
3: O que é a gruta de Notre Dame?
4: O que fica no topo do edifício principal em Notre Dame?


In [ ]:
for i in range(5):
  exemplo = dataset[i]
  pergunta_teste = exemplo['question']
  resposta_esperada = exemplo['answers']['text'][0]

  print(f"Pergunta: {pergunta_teste}")
  print(f"Resposta Esperada: {resposta_esperada}")
  print("---")

  resposta_rag = resposta_com_RAG(pergunta_teste)
  print(f"Resposta com RAG: {resposta_rag}")

  print("\n")

Pergunta: A quem a Virgem Maria supostamente apareceu em 1858 em Lourdes, na França?
Resposta Esperada: Saint Bernadette Soubirous
---
Resposta com RAG: A Virgem Maria supostamente apareceu em 1858 em Lourdes, na França para a Santa Bernadette Soubirous.


Pergunta: O que fica em frente ao edifício principal de Notre Dame?
Resposta Esperada: uma estátua de cobre de Cristo
---
Resposta com RAG: Em frente ao Edifício Principal de Notre Dame, há a Gruta de Notre Dame.


Pergunta: A Basílica do Sagrado Coração em Notre Dame fica ao lado de qual estrutura?
Resposta Esperada: o edifício principal
---
Resposta com RAG: A Basílica do Sagrado Coração em Notre Dame fica ao lado do prédio do Old College.


Pergunta: O que é a gruta de Notre Dame?
Resposta Esperada: um lugar mariano de oração e reflexão
---
Resposta com RAG: Não tenho informações específicas sobre a existência ou características de uma "gruta" na Universidade de Notre Dame. Portanto, não posso fornecer uma resposta detalhada sobre